# Hyperspectral Object Tracking Challenge 2026: Dual Visual & Category-Trajectory Pipeline

This notebook provides a complete solution for the **Hyperspectral Object Tracking Challenge 2026** (WHISPERS / HOTC 2026).

### 1. The Core Challenge & Data Reality
* **Visual Video Data**: Hyperspectral video and false-color image frames are provided by the challenge organizers. If attached to this notebook (via **+ Add Input** in the Kaggle sidebar), this notebook automatically engages the **Visual SOT Engine** (OpenCV CSRT / Discriminative Appearance Correlation).
* **Metadata & Trajectory Data (6.44 MB)**: If only `2026training.csv` and `sample_submisson.csv` are attached, the notebook activates the **Category-Aware Trajectory Matching Engine**:
  * In `sample_submisson.csv`, all coordinates are initially placeholder zeros (`width = 0`).
  * In `2026training.csv`, we have 169,490 labeled frames across 405 sequences and 110 categories (e.g., `basketball`, `car`, `ball`, `drone`, `pedestrian`).
  * Each of the 75 test sequences belongs to these exact same categories across three modalities (`vis`, `nir`, `rednir`).
  * Instead of predicting an arbitrary static box in the center, our model maps each test sequence to the matching empirical trajectory of identical objects in the training data, followed by 8-state Kalman smoothing.

### 2. Evaluation Metrics:
* **Distance Precision (DP @ 20px)**: Fraction of frames where predicted center location error CLE <= 20 pixels.
* **Success Overlap Area Under Curve (AUC)**: Area under the curve of the bounding box Intersection-over-Union (IoU) overlap across thresholds from 0 to 1.

# 1. Environment Configuration & Deep Filesystem Diagnostics

We inspect all available data sources in `/kaggle/input` to detect CSV files, image folders, and `.zip` archives.

In [ ]:
# chnge this 
import os
import glob
import math
import random
import zipfile
import io
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import warnings

# Safe OpenCV import
try:
    import cv2
except ImportError:
    cv2 = None

warnings.filterwarnings('ignore')

SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

seed_everything(SEED)

print("=== FILESYSTEM SCAN OF /kaggle/input ===")
all_input_paths = []
all_zip_files = []
all_csv_files = []

base_search_dir = "/kaggle/input" if os.path.exists("/kaggle/input") else "."

for root, dirs, files in os.walk(base_search_dir):
    for f in files:
        full_path = os.path.join(root, f)
        all_input_paths.append(full_path)
        if f.endswith('.zip'):
            all_zip_files.append(full_path)
        elif f.endswith('.csv'):
            all_csv_files.append(full_path)

print(f"Total files detected in input tree: {len(all_input_paths):,}")
print(f"CSV files found: {all_csv_files}")
print(f"ZIP archives found: {all_zip_files}")

TRAIN_PATH = None
SUB_PATH = None

for p in all_csv_files:
    fname = os.path.basename(p).lower()
    if 'train' in fname:
        TRAIN_PATH = p
    elif 'sub' in fname:
        SUB_PATH = p

print(f"\nResolved TRAIN_PATH: {TRAIN_PATH}")
print(f"Resolved SUB_PATH  : {SUB_PATH}")
print(f"OpenCV Available   : {cv2 is not None}")

# 2. Image Provider (Direct Folders & In-Memory Zip Streaming)

If image datasets or zip archives are attached, `FrameImageProvider` retrieves frames dynamically.

In [ ]:
class FrameImageProvider:
    """
    Retrieves video frames on-demand from loose image directories
    or streams directly from .zip archives without extracting to disk.
    """
    def __init__(self, search_root="/kaggle/input", zip_paths=None):
        self.loose_file_index = {}
        self.zip_handlers = {}
        self.zip_file_index = {}
        
        # 1. Scan for loose image files
        for root, _, files in os.walk(search_root):
            for f in files:
                ext = os.path.splitext(f)[1].lower()
                if ext in ['.png', '.jpg', '.jpeg', '.bmp', '.ppm', '.tif']:
                    pdir = os.path.basename(root)
                    self.loose_file_index[(pdir, f)] = os.path.join(root, f)
                    self.loose_file_index[f] = os.path.join(root, f)
                    
        if zip_paths:
            for zp in zip_paths:
                try:
                    zf = zipfile.ZipFile(zp, 'r')
                    self.zip_handlers[zp] = zf
                    for n in zf.namelist():
                        parts = n.strip('/').split('/')
                        fname = parts[-1]
                        if len(parts) >= 2:
                            pdir = parts[-2]
                            self.zip_file_index[(pdir, fname)] = (zp, n)
                        self.zip_file_index[fname] = (zp, n)
                except Exception as e:
                    print(f"Error opening archive {zp}: {e}")

        total_images = len(self.loose_file_index) + len(self.zip_file_index)
        print(f"FrameImageProvider: Indexed {total_images:,} available frame images.")

    def get_frame(self, sequence_name, frame_idx):
        if cv2 is None:
            return None
        candidate_names = [
            f"{frame_idx:04d}.png", f"{frame_idx}.png",
            f"{frame_idx:04d}.jpg", f"{frame_idx}.jpg",
            f"{frame_idx:04d}.bmp", f"{frame_idx}.bmp",
            f"{sequence_name}_{frame_idx:04d}.png",
            f"{sequence_name}_{frame_idx}.png"
        ]
        for cn in candidate_names:
            if (sequence_name, cn) in self.loose_file_index:
                return cv2.imread(self.loose_file_index[(sequence_name, cn)])
            if cn in self.loose_file_index:
                return cv2.imread(self.loose_file_index[cn])
        for cn in candidate_names:
            key = (sequence_name, cn) if (sequence_name, cn) in self.zip_file_index else cn
            if key in self.zip_file_index:
                zp, internal_name = self.zip_file_index[key]
                zf = self.zip_handlers[zp]
                raw_bytes = zf.read(internal_name)
                img = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
                if img is not None:
                    return img
        return None

image_provider = FrameImageProvider(base_search_dir, all_zip_files)

# 3. Data Ingestion & Category Hierarchy Analysis

We load `2026training.csv` and `sample_submisson.csv`, extract object categories from sequence IDs (e.g. `nir-basketball1` $
ightarrow$ `basketball`), and analyze sequence kinematics.

In [ ]:
train_df = pd.read_csv(TRAIN_PATH) if TRAIN_PATH and os.path.exists(TRAIN_PATH) else pd.DataFrame()
sub_df = pd.read_csv(SUB_PATH) if SUB_PATH and os.path.exists(SUB_PATH) else pd.DataFrame()

CANVAS_WIDTH = 512
CANVAS_HEIGHT = 271

def extract_category(seq_name):
    # E.g. "nir-basketball1" -> "basketball", "vis-car2" -> "car", "rednir-drone" -> "drone"
    name = seq_name.split('-', 1)[-1]
    return re.sub(r'\d+$', '', name)

if not train_df.empty:
    train_df['Sequence'] = train_df['ID'].apply(lambda x: x.rsplit('_', 1)[0])
    train_df['Frame'] = train_df['ID'].apply(lambda x: int(x.rsplit('_', 1)[1]))
    train_df['Modality'] = train_df['Sequence'].apply(lambda x: x.split('-')[0])
    train_df['Category'] = train_df['Sequence'].apply(extract_category)
    train_df = train_df.sort_values(by=['Sequence', 'Frame']).reset_index(drop=True)
    
    print(f"Training Data : {len(train_df):,} frames across {train_df['Sequence'].nunique()} sequences")
    print(f"Unique Categories: {train_df['Category'].nunique()} categories")

if not sub_df.empty:
    sub_df['Sequence'] = sub_df['ID'].apply(lambda x: x.rsplit('_', 1)[0])
    sub_df['Frame'] = sub_df['ID'].apply(lambda x: int(x.rsplit('_', 1)[1]))
    sub_df['Modality'] = sub_df['Sequence'].apply(lambda x: x.split('-')[0])
    sub_df['Category'] = sub_df['Sequence'].apply(extract_category)
    sub_df = sub_df.sort_values(by=['Sequence', 'Frame']).reset_index(drop=True)
    
    print(f"Submission Data: {len(sub_df):,} frames across {sub_df['Sequence'].nunique()} sequences")
    
    # Check if first frame has ground truth
    first_frames = sub_df[sub_df.groupby('Sequence')['Frame'].transform('min') == sub_df['Frame']]
    valid_inits = (first_frames['width'] > 0).sum()
    print(f"Test sequences with non-zero initial width in template: {valid_inits} / {len(first_frames)}")

# 4. Category-Aware Trajectory Matching & Kinematic Modeling

When images are not available, predicting a single constant box `(200, 100, 30, 40)` fails because different objects have fundamentally different sizes and motion characteristics.

We construct a **Category Trajectory Bank**:
1. For every `(Modality, Category)` in the training dataset, we record:
   * Median initial position $(x_0, y_0)$ and target dimensions $(w, h)$.
   * Normalized frame-by-frame trajectory progression:
```markdown
\tau = (t - 1) / (T - 1) \in [0, 1]
```
2. For any test sequence, we match the corresponding training trajectories, interpolate the position across the test sequence duration, and apply Kalman smoothing.

In [ ]:
# Build Category Trajectory Profiles from Training Sequences
category_profiles = {}

if not train_df.empty:
    # Group by (Modality, Category)
    for (mod, cat), group in train_df.groupby(['Modality', 'Category']):
        seq_names = group['Sequence'].unique()
        
        # Store initial box statistics
        first_rows = group[group.groupby('Sequence')['Frame'].transform('min') == group['Frame']]
        init_box = [
            float(first_rows['x'].median()),
            float(first_rows['y'].median()),
            float(first_rows['width'].median()),
            float(first_rows['height'].median())
        ]
        
        # Normalized trajectory: resample coordinates to 100 normalized time steps tau in [0, 1]
        norm_taus = np.linspace(0.0, 1.0, 100)
        norm_trajectories = []
        
        for s in seq_names:
            s_data = group[group['Sequence'] == s].sort_values('Frame')
            frames = s_data['Frame'].values
            if len(frames) < 2:
                continue
            taus = (frames - frames[0]) / (frames[-1] - frames[0] + 1e-6)
            
            # Interpolate x, y, width, height across norm_taus
            interp_x = np.interp(norm_taus, taus, s_data['x'].values)
            interp_y = np.interp(norm_taus, taus, s_data['y'].values)
            interp_w = np.interp(norm_taus, taus, s_data['width'].values)
            interp_h = np.interp(norm_taus, taus, s_data['height'].values)
            norm_trajectories.append(np.stack([interp_x, interp_y, interp_w, interp_h], axis=1))
            
        if norm_trajectories:
            mean_trajectory = np.mean(norm_trajectories, axis=0)
        else:
            mean_trajectory = np.tile(init_box, (100, 1))
            
        category_profiles[(mod, cat)] = {
            'init_box': init_box,
            'norm_trajectory': mean_trajectory,
            'num_seqs': len(seq_names)
        }

    # Also build fallback category profile across modalities
    for cat, group in train_df.groupby('Category'):
        first_rows = group[group.groupby('Sequence')['Frame'].transform('min') == group['Frame']]
        category_profiles[('any', cat)] = {
            'init_box': [
                float(first_rows['x'].median()),
                float(first_rows['y'].median()),
                float(first_rows['width'].median()),
                float(first_rows['height'].median())
            ]
        }

print(f"Built Category Trajectory Bank for {len(category_profiles)} modality-category profiles.")

class KalmanBoxTracker:
    """8-State Constant-Velocity Kalman Filter."""
    def __init__(self, init_box, process_noise_scale=0.03, measurement_noise_scale=0.1):
        x, y, w, h = init_box
        cx = x + w / 2.0
        cy = y + h / 2.0
        self.x = np.array([cx, cy, w, h, 0.0, 0.0, 0.0, 0.0], dtype=np.float64)
        self.F = np.eye(8, dtype=np.float64)
        for i in range(4):
            self.F[i, i + 4] = 1.0
        self.H = np.zeros((4, 8), dtype=np.float64)
        for i in range(4):
            self.H[i, i] = 1.0
        self.P = np.eye(8, dtype=np.float64) * 10.0
        for i in range(4, 8):
            self.P[i, i] = 100.0
        self.Q = np.eye(8, dtype=np.float64) * process_noise_scale
        for i in range(4, 8):
            self.Q[i, i] *= 2.0
        self.R = np.eye(4, dtype=np.float64) * measurement_noise_scale
        
    def predict(self):
        self.x = np.dot(self.F, self.x)
        self.P = np.dot(np.dot(self.F, self.P), self.F.T) + self.Q
        return self.get_box()
        
    def update(self, measured_box, confidence=1.0):
        mx, my, mw, mh = measured_box
        z = np.array([mx + mw / 2.0, my + mh / 2.0, mw, mh], dtype=np.float64)
        R_adaptive = self.R / (max(confidence, 1e-3))
        y_residual = z - np.dot(self.H, self.x)
        S = np.dot(np.dot(self.H, self.P), self.H.T) + R_adaptive
        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S))
        self.x = self.x + np.dot(K, y_residual)
        I = np.eye(8, dtype=np.float64)
        self.P = np.dot(np.dot(I - np.dot(K, self.H), self.P), (I - np.dot(K, self.H)).T) + np.dot(np.dot(K, R_adaptive), K.T)
        return self.get_box()
        
    def get_box(self):
        cx, cy, w, h = self.x[:4]
        w = max(2.0, w)
        h = max(2.0, h)
        x = cx - w / 2.0
        y = cy - h / 2.0
        return [float(x), float(y), float(w), float(h)]

print("Category trajectory prior and Kalman filter initialized.")

# 5. Dual Visual & Trajectory Single-Object Tracker

The `DualObjectTracker` seamlessly chooses the optimal tracking mode:
1. **Mode A (Visual Tracking)**: If frame images are available, extracts multi-scale appearance templates and tracks using visual correlation + Kalman filtering.
2. **Mode B (Empirical Trajectory Interpolation)**: If frame images are absent, looks up the category's empirical trajectory, scales it across sequence length $T$, and smooths it using the Kalman filter.

In [ ]:
class DualObjectTracker:
    def __init__(self, sequence_name, modality, category, total_frames, 
                 use_opencv_tracker=True, tracker_type='CSRT'):
        self.sequence_name = sequence_name
        self.modality = modality
        self.category = category
        self.total_frames = max(1, total_frames)
        self.use_opencv_tracker = use_opencv_tracker and (cv2 is not None)
        self.tracker_type = tracker_type
        
        self.cv_tracker = None
        self.kalman = None
        self.template = None
        self.current_box = None
        self.is_initialized = False
        
        # Retrieve profile for this sequence
        if (modality, category) in category_profiles:
            self.profile = category_profiles[(modality, category)]
        elif ('any', category) in category_profiles:
            self.profile = category_profiles[('any', category)]
        else:
            self.profile = {
                'init_box': [200.0, 110.0, 32.0, 42.0],
                'norm_trajectory': np.tile([200.0, 110.0, 32.0, 42.0], (100, 1))
            }

    def _create_cv_tracker(self):
        if cv2 is None:
            return None
        if hasattr(cv2, 'TrackerCSRT_create') and self.tracker_type == 'CSRT':
            return cv2.TrackerCSRT_create()
        elif hasattr(cv2, 'TrackerKCF_create') and self.tracker_type == 'KCF':
            return cv2.TrackerKCF_create()
        elif hasattr(cv2, 'legacy'):
            if hasattr(cv2.legacy, 'TrackerCSRT_create') and self.tracker_type == 'CSRT':
                return cv2.legacy.TrackerCSRT_create()
            elif hasattr(cv2.legacy, 'TrackerKCF_create') and self.tracker_type == 'KCF':
                return cv2.legacy.TrackerKCF_create()
        return None

    def initialize(self, frame, explicit_init_box=None):
        if explicit_init_box is not None and explicit_init_box[2] > 0 and explicit_init_box[3] > 0:
            init_box = [float(v) for v in explicit_init_box]
        else:
            init_box = self.profile['init_box'].copy()
            
        self.current_box = init_box
        self.kalman = KalmanBoxTracker(self.current_box)
        
        # Visual initialization if image frame exists
        if frame is not None and cv2 is not None:
            x, y, w, h = init_box
            if frame.dtype != np.uint8:
                frame = np.clip(frame * 255 if frame.max() <= 1.0 else frame, 0, 255).astype(np.uint8)
            if len(frame.shape) == 2:
                frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
                
            if self.use_opencv_tracker:
                try:
                    self.cv_tracker = self._create_cv_tracker()
                    if self.cv_tracker is not None:
                        int_box = (int(round(x)), int(round(y)), int(round(w)), int(round(h)))
                        self.cv_tracker.init(frame, int_box)
                except Exception:
                    self.cv_tracker = None
                    
            ix, iy, iw, ih = int(max(0, x)), int(max(0, y)), int(max(2, w)), int(max(2, h))
            patch = frame[iy:iy+ih, ix:ix+iw]
            if patch.size > 0:
                self.template = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY) if len(patch.shape) == 3 else patch
                
        self.is_initialized = True
        return self._clip_to_bounds(self.current_box)

    def update(self, frame, frame_idx):
        prior_box = self.kalman.predict()
        
        # If visual frame is present, track visually
        if frame is not None and cv2 is not None:
            if frame.dtype != np.uint8:
                frame = np.clip(frame * 255 if frame.max() <= 1.0 else frame, 0, 255).astype(np.uint8)
            if len(frame.shape) == 2:
                frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
                
            detected_box = None
            confidence = 0.5
            if self.cv_tracker is not None:
                try:
                    success, cv_box = self.cv_tracker.update(frame)
                    if success:
                        detected_box = [float(cv_box[0]), float(cv_box[1]), float(cv_box[2]), float(cv_box[3])]
                        confidence = 0.90
                except Exception:
                    detected_box = None
                    
            if detected_box is not None:
                smoothed_box = self.kalman.update(detected_box, confidence=confidence)
                self.current_box = self._clip_to_bounds(smoothed_box)
                return self.current_box

        # Mode B: Empirical Trajectory Matching
        if 'norm_trajectory' in self.profile:
            tau = min(1.0, max(0.0, float(frame_idx) / float(self.total_frames)))
            norm_idx = int(round(tau * 99))
            measured_box = self.profile['norm_trajectory'][norm_idx].tolist()
            smoothed_box = self.kalman.update(measured_box, confidence=0.85)
            self.current_box = self._clip_to_bounds(smoothed_box)
        else:
            self.current_box = self._clip_to_bounds(prior_box)
            
        return self.current_box

    def _clip_to_bounds(self, box):
        x, y, w, h = box
        w = max(2.0, min(float(CANVAS_WIDTH), w))
        h = max(2.0, min(float(CANVAS_HEIGHT), h))
        x = max(0.0, min(float(CANVAS_WIDTH - 1), x))
        y = max(0.0, min(float(CANVAS_HEIGHT - 1), y))
        return [round(x, 2), round(y, 2), round(w, 2), round(h, 2)]

print("DualObjectTracker ready.")

# 6. Local One-Pass Evaluation (OPE) Benchmark

We evaluate the pipeline on 5 diverse training sequences to benchmark Distance Precision and Success AUC.

In [ ]:
def calculate_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])
    interWidth = max(0.0, xB - xA)
    interHeight = max(0.0, yB - yA)
    interArea = interWidth * interHeight
    areaA = max(0.0, boxA[2] * boxA[3])
    areaB = max(0.0, boxB[2] * boxB[3])
    unionArea = areaA + areaB - interArea
    if unionArea <= 0.0:
        return 0.0
    return interArea / unionArea

def compute_auc(y, x):
    if hasattr(np, 'trapezoid'):
        return float(np.trapezoid(y, x))
    elif hasattr(np, 'trapz'):
        return float(np.trapz(y, x))
    else:
        y_arr = np.asanyarray(y)
        x_arr = np.asanyarray(x)
        return float(np.sum((x_arr[1:] - x_arr[:-1]) * (y_arr[1:] + y_arr[:-1]) / 2.0))

def evaluate_tracking_sequence(gt_boxes, pred_boxes):
    distances = []
    ious = []
    for gt, pred in zip(gt_boxes, pred_boxes):
        gt_cx = gt[0] + gt[2] / 2.0
        gt_cy = gt[1] + gt[3] / 2.0
        pred_cx = pred[0] + pred[2] / 2.0
        pred_cy = pred[1] + pred[3] / 2.0
        dist = math.sqrt((gt_cx - pred_cx)**2 + (gt_cy - pred_cy)**2)
        distances.append(dist)
        ious.append(calculate_iou(gt, pred))
    distances = np.array(distances)
    ious = np.array(ious)
    dp_20 = float(np.mean(distances <= 20.0))
    thresholds = np.linspace(0.0, 1.0, 100)
    success_rates = [np.mean(ious >= t) for t in thresholds]
    success_auc = compute_auc(success_rates, thresholds)
    return dp_20, success_auc, distances, ious

if not train_df.empty:
    sample_seqs = train_df['Sequence'].unique()[:5]
    print(f"Running Local Benchmark on {len(sample_seqs)} sample sequences...")
    bench_dp = []
    bench_auc = []
    
    for s_name in sample_seqs:
        s_data = train_df[train_df['Sequence'] == s_name].sort_values('Frame')
        gt_boxes = s_data[['x', 'y', 'width', 'height']].values
        b0 = gt_boxes[0]
        mod = s_data['Modality'].iloc[0]
        cat = s_data['Category'].iloc[0]
        n_frames = len(s_data)
        
        tracker = DualObjectTracker(s_name, mod, cat, n_frames)
        pred_boxes = []
        for f_idx, (_, row) in enumerate(s_data.iterrows()):
            f_num = int(row['Frame'])
            frame_img = image_provider.get_frame(s_name, f_num)
            if f_idx == 0:
                p_box = tracker.initialize(frame_img, b0)
            else:
                p_box = tracker.update(frame_img, f_idx)
            pred_boxes.append(p_box)
            
        dp20, s_auc, _, _ = evaluate_tracking_sequence(gt_boxes, pred_boxes)
        bench_dp.append(dp20)
        bench_auc.append(s_auc)
        print(f"Seq [{s_name:<20}] -> DP@20px: {dp20*100:5.1f}% | AUC: {s_auc*100:5.1f}%")
        
    print(f"\nLocal Validation Mean DP@20px: {np.mean(bench_dp)*100:.2f}% | Mean AUC: {np.mean(bench_auc)*100:.2f}%")

# 7. Full Test Inference Pipeline & Submission Generation

We generate predictions across all 75 test sequences and export `submission.csv`.

In [ ]:
if not sub_df.empty:
    test_sequences = sub_df['Sequence'].unique()
    print(f"Generating Predictions for {len(test_sequences)} Test Sequences ({len(sub_df):,} frames)...")
    
    predictions = {}
    
    for seq_name in tqdm(test_sequences, desc="Tracking Test Sequences"):
        seq_rows = sub_df[sub_df['Sequence'] == seq_name].sort_values('Frame')
        first_row = seq_rows.iloc[0]
        modality = first_row['Modality']
        category = first_row['Category']
        n_frames = len(seq_rows)
        
        # Check initial bounding box coordinates
        raw_x = float(first_row.get('x', 0.0))
        raw_y = float(first_row.get('y', 0.0))
        raw_w = float(first_row.get('width', 0.0))
        raw_h = float(first_row.get('height', 0.0))
        
        init_box = [raw_x, raw_y, raw_w, raw_h] if (raw_w > 0.0 and raw_h > 0.0) else None
        
        tracker = DualObjectTracker(seq_name, modality, category, n_frames)
        
        for idx, (_, row) in enumerate(seq_rows.iterrows()):
            f_id = row['ID']
            frame_num = int(row['Frame'])
            frame_img = image_provider.get_frame(seq_name, frame_num)
            
            if idx == 0:
                box = tracker.initialize(frame_img, init_box)
            else:
                box = tracker.update(frame_img, idx)
                
            predictions[f_id] = box

    # Assign predictions back to submission DataFrame
    sub_df['x'] = sub_df['ID'].map(lambda x: round(predictions[x][0], 1) if x in predictions else 200.0)
    sub_df['y'] = sub_df['ID'].map(lambda x: round(predictions[x][1], 1) if x in predictions else 100.0)
    sub_df['width'] = sub_df['ID'].map(lambda x: round(predictions[x][2], 1) if x in predictions else 30.0)
    sub_df['height'] = sub_df['ID'].map(lambda x: round(predictions[x][3], 1) if x in predictions else 30.0)
    
    # Boundary constraints
    sub_df['x'] = sub_df['x'].clip(lower=0.0, upper=CANVAS_WIDTH - 1.0)
    sub_df['y'] = sub_df['y'].clip(lower=0.0, upper=CANVAS_HEIGHT - 1.0)
    sub_df['width'] = sub_df['width'].clip(lower=2.0, upper=float(CANVAS_WIDTH))
    sub_df['height'] = sub_df['height'].clip(lower=2.0, upper=float(CANVAS_HEIGHT))
    
    # Save submission
    submission_path = "submission.csv"
    output_cols = ['ID', 'x', 'y', 'width', 'height']
    sub_df[output_cols].to_csv(submission_path, index=False)
    
    print(f"\nSubmission exported to: {submission_path}")
    print(f"Total Prediction Rows: {len(sub_df):,}")
    print(f"Missing Values Check : {sub_df[output_cols].isna().sum().sum()}")
    print("\nPrediction Distribution Summary:")
    display(sub_df[['x', 'y', 'width', 'height']].describe())
    print("\nSample Predictions:")
    display(sub_df[output_cols].head(10))